In [1]:
import pandas as pd
from omicsqc.ingestion import ingest_data
from omicsqc.preprocessing.transform import transform_data
from omicsqc.qc_layer.qc_metrics import compute_qc_individual
from omicsqc.qc_layer.qc_metrics import compute_qc_pipeline
from omicsqc.preprocessing.preprocess import preprocess_pipeline

In [2]:
data_df = pd.read_csv("data/ProteinAbundances.csv")
meta_df = pd.read_csv("data/meta_data.csv")
protdata = ingest_data(data_df, meta_df,)

[Success] Found 558 overlapping technical samples.


### Original ingested data

In [3]:
protdata = compute_qc_pipeline(protdata, data_df= protdata.data,meta_df=protdata.metadata, protein_threshold_pct= 40, 
                               variation_factors= ["Batch_overall", "JohnsonDiagnosis", "cogdx", "braaksc", "niareagansc", "msex"])

22.337005684771253


In [4]:
protdata.logs

['[Missingness Analysis]\n        Per-protein missingness (mean): 22.34%\n        Per-sample missingness (mean):  22.34%\n        Overall missingness:            22.34%\n        Spearman coefficient:           0.8788\n        Spearman p-value:               0.0000e+00\n        Inferred missingness nature: Evidence of intensity-dependent missingness, consistent with left-censoring',
 '[Batch Variation Test]\n    Statistical method:             anova \n\n    Significance level (alpha):     0.05 \n\n    Protein threshold:              40.00% \n\n    Significant proteins:           10586 \n\n    Significant proteins (%):       88.53% \n\n    Batch effect detected:          True \n\n    silhoutte score:                0.0101',
 '[Explained Variance Comparison]\n    Factors analyzed:               Batch_overall, JohnsonDiagnosis, cogdx, braaksc, niareagansc, msex\n    Per-protein comparison shape:   (11958, 6)\n    Summary dataframe shape:        (6, 7)']

### Running preprocessing pipeline on the protdata 

In [5]:
processed_protdata = preprocess_pipeline(protdata, 40, 60, imputation="MinProb", batch_col="Batch_overall")

### Assessing qc metrics on the transformed data

In [6]:
temp_report = compute_qc_individual(processed_protdata.transformed_data, processed_protdata.filtered_metadata,
                                    protein_threshold_pct=40, variation_factors= ["Batch_overall", "JohnsonDiagnosis", "cogdx", "braaksc", "niareagansc", "msex"])

0.00016882851581163463


c:\Users\Khalid Mohammad\Projects\omicsqc\.venv\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log2
  result = func(self.values, **kwargs)


In [7]:
temp_report.logs

['[Missingness Analysis]\n        Per-protein missingness (mean): 0.00%\n        Per-sample missingness (mean):  0.00%\n        Overall missingness:            0.00%\n        Spearman coefficient:           0.8867\n        Spearman p-value:               0.0000e+00\n        Inferred missingness nature: Evidence of intensity-dependent missingness, consistent with left-censoring',
 '[Batch Variation Test]\n    Statistical method:             anova \n\n    Significance level (alpha):     0.05 \n\n    Protein threshold:              40.00% \n\n    Significant proteins:           6369 \n\n    Significant proteins (%):       100.00% \n\n    Batch effect detected:          True \n\n    silhoutte score:                0.6296',
 '[Explained Variance Comparison]\n    Factors analyzed:               Batch_overall, JohnsonDiagnosis, cogdx, braaksc, niareagansc, msex\n    Per-protein comparison shape:   (6369, 6)\n    Summary dataframe shape:        (6, 7)']

### Assessing qc metrics on the imputed data

In [8]:
processed_protdata = compute_qc_pipeline(protdata= processed_protdata, data_df=processed_protdata.imputed_data,
                                         meta_df=processed_protdata.filtered_metadata, protein_threshold_pct= 40, 
                               variation_factors= ["Batch_overall", "JohnsonDiagnosis", "cogdx", "braaksc", "niareagansc", "msex"])
processed_protdata.logs

0.0


['missingness analysis not applicable because no missing values remain',
 '[Batch Variation Test]\n    Statistical method:             anova \n\n    Significance level (alpha):     0.05 \n\n    Protein threshold:              40.00% \n\n    Significant proteins:           6369 \n\n    Significant proteins (%):       100.00% \n\n    Batch effect detected:          True \n\n    silhoutte score:                0.6302',
 '[Explained Variance Comparison]\n    Factors analyzed:               Batch_overall, JohnsonDiagnosis, cogdx, braaksc, niareagansc, msex\n    Per-protein comparison shape:   (6369, 6)\n    Summary dataframe shape:        (6, 7)']

### Sanity check between transformed data and imputed data

In [9]:
print(processed_protdata.transformed_data is processed_protdata.imputed_data)

False


In [10]:
print(processed_protdata.transformed_data.isna().sum().sum())
print(processed_protdata.imputed_data.isna().sum().sum())

6
0


The filter function removed proteins that were completely missing across batches or proteins with missingness above the threshold across all samples. Hence why transformed data has only 6 NaNs. The imputation thus did not have significant impact on the data distribution as showcased below by comparing both dataframe's modes of central tendencies and quartile ranges

In [11]:
protdata.transformed_data.isna().sum().sum()

np.int64(6)

In [12]:
protdata.transformed_data.describe()

,rmap_b41.130C,rmap_b26.128N,rmap_b13.130C,bnnr_b20.128N,rmap_b47.127C,bnnr_b16.127C,rmap_b08.129C,bnnr_b07.130C,rmap_b39.128C,rmap_b39.128N,...,rmap_b49.130N,rmap_b50.128N,rmap_b46.127N,rmap_b36.130C,rmap_b15.130N,bnnr_b18.130C,rmap_b38.128N,rmap_b35.128N,rmap_b39.129N,rmap_b13.130N
count,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,...,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000
mean,0.157835,0.237473,0.228590,0.288184,0.246491,0.225166,0.221994,0.246593,0.174708,0.196963,...,0.190904,0.179105,0.079418,0.229992,0.269279,0.263016,0.237106,0.284187,0.182591,0.249994
std,2.852581,2.274949,2.398888,2.659547,2.370119,2.675810,2.297435,2.655164,2.710571,2.740390,...,2.373143,2.467519,2.921796,2.528134,2.296987,2.653788,2.606385,2.452321,2.715355,2.388977
min,-9.610960,-8.041121,-10.115987,-8.268496,-5.810839,-7.318281,-8.023355,-7.581348,-8.491028,-8.859381,...,-7.306054,-8.957183,-9.551052,-9.231246,-7.442270,-6.641947,-8.251506,-9.789083,-8.837908,-9.152386
25%,-1.903667,-1.451004,-1.507265,-1.620939,-1.491869,-1.730040,-1.368843,-1.705370,-1.822197,-1.829115,...,-1.539297,-1.599218,-2.041576,-1.614452,-1.417604,-1.682354,-1.648058,-1.520918,-1.822998,-1.454721
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.969410,1.659931,1.690327,1.847088,1.720686,1.813131,1.620121,1.817154,1.873167,1.914701,...,1.691002,1.740772,2.016804,1.795658,1.636986,1.851660,1.849768,1.805321,1.889168,1.699867
max,9.472057,8.656363,8.808218,11.131729,9.211075,10.572899,8.504427,10.398933,9.567450,9.696070,...,9.008171,8.792404,9.117624,9.174131,10.164957,10.908045,9.826931,9.291471,9.605986,9.878719


In [13]:
protdata.imputed_data.describe()

,rmap_b41.130C,rmap_b26.128N,rmap_b13.130C,bnnr_b20.128N,rmap_b47.127C,bnnr_b16.127C,rmap_b08.129C,bnnr_b07.130C,rmap_b39.128C,rmap_b39.128N,...,rmap_b49.130N,rmap_b50.128N,rmap_b46.127N,rmap_b36.130C,rmap_b15.130N,bnnr_b18.130C,rmap_b38.128N,rmap_b35.128N,rmap_b39.129N,rmap_b13.130N
count,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,...,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000,6369.000000
mean,0.157835,0.237473,0.228590,0.288184,0.246491,0.225166,0.221994,0.246593,0.174708,0.196963,...,0.190904,0.179105,0.079418,0.229992,0.269279,0.263016,0.237106,0.284187,0.182592,0.249994
std,2.852581,2.274949,2.398888,2.659547,2.370119,2.675810,2.297435,2.655164,2.710571,2.740390,...,2.373143,2.467519,2.921796,2.528134,2.296987,2.653788,2.606385,2.452321,2.715355,2.388977
min,-9.610960,-8.041121,-10.115987,-8.268496,-5.810839,-7.318281,-8.023355,-7.581348,-8.491028,-8.859381,...,-7.306054,-8.957183,-9.551052,-9.231246,-7.442270,-6.641947,-8.251506,-9.789083,-8.837908,-9.152386
25%,-1.903667,-1.451004,-1.507265,-1.620939,-1.491869,-1.730040,-1.368843,-1.705370,-1.822197,-1.829115,...,-1.539297,-1.599218,-2.041576,-1.614452,-1.417604,-1.682354,-1.648058,-1.520918,-1.822998,-1.454721
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.969410,1.659931,1.690327,1.847088,1.720686,1.813131,1.620121,1.817154,1.873167,1.914701,...,1.691002,1.740772,2.016804,1.795658,1.636986,1.851660,1.849768,1.805321,1.889168,1.699867
max,9.472057,8.656363,8.808218,11.131729,9.211075,10.572899,8.504427,10.398933,9.567450,9.696070,...,9.008171,8.792404,9.117624,9.174131,10.164957,10.908045,9.826931,9.291471,9.605986,9.878719


In [14]:
processed_protdata.invalid_proteins

['|A6NIZ1',
 '|A6NNZ2',
 '|P00761',
 '|P02534',
 '|P15241',
 '|P25691',
 '|Q56UQ5',
 '|Q6ZSR9',
 '|Q9H8V8',
 'A1BG|P04217',
 'A2M|P01023',
 'AAAS|Q9NRG9',
 'AACS|Q86V21',
 'AADAT|Q8N5Z0',
 'AAGAB|Q6PD74',
 'AAK1|Q2M2I8',
 'AAMDC|Q9H7C9',
 'AAMP|Q13685',
 'AAR2|Q9Y312',
 'AARS|P49588',
 'AARS2|Q5JTZ9',
 'AARSD1|Q9BTE6',
 'AASDHPPT|Q9NRN7',
 'AASS|Q9UDR5',
 'AATK|Q6ZMQ8',
 'ABAT|P80404',
 'ABCA1|O95477',
 'ABCA10|Q8WWZ4',
 'ABCA2|Q9BZC7',
 'ABCA3|Q99758',
 'ABCA5|Q8WWZ7',
 'ABCA7|Q8IZY2',
 'ABCA8|O94911',
 'ABCA9|Q8IUA7',
 'ABCB1|P08183',
 'ABCB10|Q9NRK6',
 'ABCB4|P21439',
 'ABCB5|Q2M3G0',
 'ABCB6|Q9NP58',
 'ABCB7|O75027',
 'ABCB8|Q9NUT2',
 'ABCB9|Q9NP78',
 'ABCC1|P33527',
 'ABCC4|O15439',
 'ABCC5|O15440',
 'ABCC8|Q09428',
 'ABCC9|O60706',
 'ABCD1|P33897',
 'ABCD2|Q9UBJ2',
 'ABCD3|P28288',
 'ABCE1|P61221',
 'ABCF1|Q8NE71',
 'ABCF2|Q9UG63',
 'ABCF3|Q9NUQ8',
 'ABCG1|P45844',
 'ABCG2|Q9UNQ0',
 'ABCG4|Q9H172',
 'ABHD10|Q9NUJ1',
 'ABHD11|Q8NFV4',
 'ABHD12|Q8N2K0',
 'ABHD13|Q7L211',
 'ABHD14A|